In [27]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import os,  numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [4]:
# Input WITHOUT PCA (your raw master file)
RAW_IN_PATH = "cleaned_trade_master.csv"

In [5]:
# Output WITH PCA (descriptive names applied)
PCA_OUT_PATH = "cleaned_trade_master_with_pca.csv"

In [6]:
# Directory for clustering outputs
OUT_DIR = Path("cluster_outputs")

In [7]:
# K-Means search range
K_RANGE = range(2, 10)
RANDOM_STATE = 42

In [8]:
# Hierarchical (optional)
MAX_DENDRO_LABELS = 60
HIER_CUT_N = None   # e.g., 4 to also save hierarchical labels; leave None to skip label CSV

In [9]:
# Aggregation mode for PCA vectors at Country/Product level
AGG_METHOD = "weighted"   # "mean" or "weighted"
WEIGHT_COL = "Value_2023" # weight for weighted mean

In [10]:
# Labeling controls
TOP_N_COUNTRIES = 20
TOP_N_PRODUCTS  = 20
ALSO_LABEL_COUNTRIES = []   # e.g., ["Qatar","Bahrain","Oman"]
ALSO_LABEL_PRODUCTS  = []   # e.g., ["Petroleum oils","Gold"]

In [11]:
# Descriptive names mapping for PC1..PC10
PC_NAME_MAP = {
    "PC1": "Overall_Trade_Scale_and_Market_Share",
    "PC2": "Trade_Profile_and_Market_Orientation",
    "PC3": "Growth_and_Competitiveness",
    "PC4": "Global_Position_and_Logistics_Factor",
    "PC5": "Tariff_and_Trade_Policy_Impact",
    "PC6": "Distance_Adjusted_Growth_Effect",
    "PC7": "Regional_Concentration_Dynamics",
    "PC8": "Market_Share_Sensitivity",
    "PC9": "Policy_Driven_Trade_Variation",
    "PC10": "Tariff_Residual_Effect",
}

# Axis names for the 2D projection (PCA on aggregated features)
AXIS1_NAME = "Aggregated_Trade_Profile_Axis1"
AXIS2_NAME = "Aggregated_Trade_Profile_Axis2"

In [12]:
# ==========================
# Utilities
# ==========================
def safe_require_cols(df: pd.DataFrame, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Required column(s) missing: {missing}")

In [13]:
def find_pca_columns(df: pd.DataFrame):
    """
    Return list of PCA columns present in df, preserving order.
    Works whether columns are named PC1..PCn or with descriptive names.
    """
    # Classic names (in case any survived rename)
    classic_pc_order = [f"PC{i}" for i in range(1, 101)]
    # Descriptive names (the ones we rename to)
    descriptive_pc_order = list(PC_NAME_MAP.values())

    classic_found = [c for c in classic_pc_order if c in df.columns]
    if classic_found:
        return classic_found

    descriptive_found = [c for c in descriptive_pc_order if c in df.columns]
    if descriptive_found:
        return descriptive_found

    # Fallback: guess PCA cols (numeric & not obvious base features)
    likely_keys = {
        "Country","Product","Code","Year","Value_2023",
        "Trade_Balance_2023","Growth_2019_2023","Growth_2022_2023",
        "World_Growth","World_Import_Rank","Avg_Distance_km",
        "Concentration","World_Import_Share","Avg_Tariff","Direction",
        "WEIGHT","Weight","Total"
    }
    numeric_cols = df.select_dtypes(include="number").columns
    guessed = [c for c in numeric_cols if c not in likely_keys]
    if guessed:
        return guessed

    raise ValueError("No PCA columns found (PCi or descriptive).")

In [14]:
def aggregate_pcs(df, key_col, pca_cols):
    """
    Aggregate PCA vectors by Country/Product using mean or Value_2023-weighted mean.
    Returns (names, X (scaled), totals_for_labeling).
    """
    if AGG_METHOD == "weighted":
        if WEIGHT_COL not in df.columns:
            raise ValueError(f"Weighted mode requires column '{WEIGHT_COL}'.")
        def wavg(g):
            w = g[WEIGHT_COL].replace(0, np.nan).fillna(0.0)
            return (g[pca_cols].multiply(w, axis=0)).sum() / (w.sum() if w.sum() != 0 else 1.0)
        grp = df.groupby(key_col, as_index=True).apply(wavg)
    else:
        grp = df.groupby(key_col, as_index=True)[pca_cols].mean()

    X = StandardScaler().fit_transform(grp.values)
    names = grp.index.to_list()
    totals = df.groupby(key_col)[WEIGHT_COL].sum().reindex(names) if WEIGHT_COL in df.columns else pd.Series([1]*len(names), index=names)
    return names, X, totals

In [15]:
def optimal_kmeans_k(X):
    best_k, best_s, scores = None, -1.0, []
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X)
        s = silhouette_score(X, labels)
        scores.append((k, s))
        if s > best_s:
            best_k, best_s = k, s
    return best_k, best_s, scores

In [16]:
def plot_silhouette(scores, title, out_png):
    ks = [k for k, _ in scores]
    ss = [s for _, s in scores]
    plt.figure(figsize=(6,4))
    plt.plot(ks, ss, marker="o")
    plt.title(title + " (best K auto-selected)")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Silhouette score")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()

In [17]:
def pca2(X):
    """PCA to 2D for plotting on aggregated feature space."""
    return PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

In [18]:
def coords_lookup(names, labels, X2, level_name, out_csv):
    # These 2D coords are a PCA of the aggregated feature space (not original PCs)
    pd.DataFrame({
        level_name: names,
        "Cluster": labels,
        AXIS1_NAME: X2[:,0],
        AXIS2_NAME: X2[:,1]
    }).to_csv(out_csv, index=False)

In [19]:
def scatter_with_labels_on_demand(X2, labels, names, title, out_png,
                                  top_n=None, totals=None, also_label=None):
    names = list(map(str, names))
    label_set = set(map(str, also_label or []))

    if top_n and totals is not None:
        top_names = totals.sort_values(ascending=False).head(top_n).index
        label_set |= set(map(str, top_names))

    plt.figure(figsize=(11, 9))
    for c in sorted(np.unique(labels)):
        m = labels == c
        plt.scatter(X2[m, 0], X2[m, 1], s=26, label=f"Cluster {c}")

    for i, n in enumerate(names):
        if (not label_set) or (n in label_set):
            plt.annotate(n, (X2[i, 0], X2[i, 1]), xytext=(3, 3),
                         textcoords="offset points", fontsize=8, alpha=0.9)

    plt.title(title)
    plt.xlabel(f"{AXIS1_NAME}")
    plt.ylabel(f"{AXIS2_NAME}")
    plt.legend(markerscale=1.5, fontsize=8)
    plt.tight_layout()
    plt.savefig(out_png, dpi=180)
    plt.close()

In [20]:
def plot_per_cluster_labeled(X2, labels, names, level_name, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    names = list(map(str, names))

    for c in sorted(np.unique(labels)):
        m = labels == c
        plt.figure(figsize=(8, 6))
        plt.scatter(X2[m, 0], X2[m, 1], s=30, label=f"Cluster {c}")

        sub_names = np.array(names)[m]
        for i, n in enumerate(sub_names):
            plt.annotate(n, (X2[m, 0][i], X2[m, 1][i]), xytext=(3, 3),
                         textcoords="offset points", fontsize=8, alpha=0.9)

        plt.title(f"{level_name} — Cluster {c}")
        plt.xlabel(f"{AXIS1_NAME}")
        plt.ylabel(f"{AXIS2_NAME}")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(out_dir / f"{level_name}_cluster_{c}.png", dpi=180)
        plt.close()

In [21]:
def run_kmeans_and_plots(level_name, names, X, totals, top_n, also_label, base_out):
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # choose K
    k_opt, sil, scores = optimal_kmeans_k(X)
    plot_silhouette(scores, f"{level_name} — Silhouette vs K", OUT_DIR / f"{base_out}_silhouette.png")

    # final model
    km = KMeans(n_clusters=k_opt, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X)

    # 2D coords
    X2 = pca2(X)

    # coords lookup (with descriptive axis names)
    coords_lookup(names, labels, X2, level_name, OUT_DIR / f"{base_out}_coords_lookup_pca2d.csv")

    # scatter on demand
    scatter_with_labels_on_demand(
        X2, labels, names,
        title=f"{level_name} — K-Means (K={k_opt}) — Top {top_n} + on-demand",
        out_png=OUT_DIR / f"{base_out}_kmeans_pca2d_top{top_n}.png",
        top_n=top_n, totals=totals, also_label=also_label
    )

    # per-cluster labeled plots (all names)
    plot_per_cluster_labeled(
        X2, labels, names, level_name,
        OUT_DIR / f"{base_out}_percluster_plots"
    )

    # save labels
    pd.DataFrame({level_name: names, "KMeans_Cluster": labels}).to_csv(
        OUT_DIR / f"{base_out}_kmeans_labels.csv", index=False
    )
    return labels, X2

In [22]:
def run_hierarchical_global(level_name, names, X, base_out):
    from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

    Z = linkage(X, method="ward", metric="euclidean")

    plt.figure(figsize=(12, 7))
    if len(names) > MAX_DENDRO_LABELS:
        dendrogram(
            Z, truncate_mode="lastp", p=MAX_DENDRO_LABELS,
            show_leaf_counts=True, leaf_rotation=90.0,
        )
        plt.title(f"{level_name} — Hierarchical Dendrogram (truncated)")
        plt.xlabel(f"{level_name} groups (truncated)")
    else:
        dendrogram(Z, labels=list(map(str, names)), leaf_rotation=90.0, leaf_font_size=8)
        plt.title(f"{level_name} — Hierarchical Dendrogram")
        plt.xlabel(level_name)

    plt.ylabel("Linkage distance (Ward, Euclidean)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{base_out}_hier_dendrogram.png", dpi=180)
    plt.close()

    if HIER_CUT_N and HIER_CUT_N >= 2:
        labs = fcluster(Z, t=HIER_CUT_N, criterion="maxclust")
        pd.DataFrame({level_name: names, "Hier_Cluster": labs}).to_csv(
            OUT_DIR / f"{base_out}_hier_labels_k{HIER_CUT_N}.csv", index=False
        )

In [31]:
# ==========================
# Pipeline
# ==========================
def run_pipeline():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # --- Load raw data ---
    df = pd.read_csv(RAW_IN_PATH)

    # sanity check
    safe_require_cols(df, ["Country", "Product"])
    if AGG_METHOD == "weighted":
        safe_require_cols(df, [WEIGHT_COL])

    # --- PCA on numeric columns ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Median imputation
    imputer = SimpleImputer(strategy="median")
    df_filled = df.copy()
    df_filled[numeric_cols] = imputer.fit_transform(df[numeric_cols])

    # Scale
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_filled[numeric_cols])

    # PCA: keep 95% variance
    pca_full = PCA(n_components=0.95, random_state=RANDOM_STATE)
    pca_data = pca_full.fit_transform(scaled_data)

    # Build PCA frame dynamically (no hard-coded count)
    n_comps = getattr(pca_full, "n_components_", None) or pca_data.shape[1]
    pc_cols = [f"PC{i+1}" for i in range(n_comps)]
    df_with_pca = pd.DataFrame(pca_data, columns=pc_cols)
    df_with_pca = pd.concat([df.reset_index(drop=True), df_with_pca], axis=1)

    # Rename PCs to descriptive names (only ones that exist)
    rename_map = {k: v for k, v in PC_NAME_MAP.items() if k in df_with_pca.columns}
    df_with_pca.rename(columns=rename_map, inplace=True)

    # Save PCA master CSV
    df_with_pca.to_csv(PCA_OUT_PATH, index=False)

    # Save explained variance with both original PCi and descriptive names
    pc_list = [f"PC{i+1}" for i in range(n_comps)]
    expvar = pd.DataFrame({
        "Component_PC": pc_list,
        "Component_Descriptive": [PC_NAME_MAP.get(pc, pc) for pc in pc_list],
        "ExplainedVarianceRatio": pca_full.explained_variance_ratio_,
    })
    expvar.to_csv("cleaned_trade_master_with_pca_explained_variance.csv", index=False)

    # Save list of PCA columns used in the master (for traceability)
    present_pca_cols = [c for c in df_with_pca.columns if (c in PC_NAME_MAP.values()) or c.startswith("PC")]
    pd.DataFrame({"PCA_Columns_Present": present_pca_cols}).to_csv("pca_columns_used.csv", index=False)

    # --- Clustering on df_with_pca ---
    pca_cols = find_pca_columns(df_with_pca)

    # COUNTRY
    country_names, X_country, country_totals = aggregate_pcs(df_with_pca, "Country", pca_cols)
    c_labels, X2_country = run_kmeans_and_plots(
        "Country", country_names, X_country, country_totals,
        top_n=TOP_N_COUNTRIES, also_label=ALSO_LABEL_COUNTRIES, base_out="country"
    )
    run_hierarchical_global("Country", country_names, X_country, "country")

    # PRODUCT
    product_names, X_product, product_totals = aggregate_pcs(df_with_pca, "Product", pca_cols)
    p_labels, X2_product = run_kmeans_and_plots(
        "Product", product_names, X_product, product_totals,
        top_n=TOP_N_PRODUCTS, also_label=ALSO_LABEL_PRODUCTS, base_out="product"
    )
    run_hierarchical_global("Product", product_names, X_product, "product")

    # Meta: record configuration + detected columns (JSON)
    meta = {
        "RAW_IN_PATH": RAW_IN_PATH,
        "PCA_OUT_PATH": PCA_OUT_PATH,
        "AGG_METHOD": AGG_METHOD,
        "WEIGHT_COL": WEIGHT_COL,
        "K_RANGE": [min(K_RANGE), max(K_RANGE)],
        "RANDOM_STATE": RANDOM_STATE,
        "DescriptivePCNames": PC_NAME_MAP,
        "PCA_Columns_Present": present_pca_cols,
        "PlotAxes": {"Axis1": AXIS1_NAME, "Axis2": AXIS2_NAME},
    }
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(OUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    print("\nSaved PCA master to:", Path(PCA_OUT_PATH).resolve())
    print("Saved explained variance to:", Path('cleaned_trade_master_with_pca_explained_variance.csv').resolve())
    print("Saved PCA column list to:", Path('pca_columns_used.csv').resolve())
    print("All clustering outputs in:", OUT_DIR.resolve())
    for p in sorted(OUT_DIR.glob("*")):
        print(" -", p.name)

if __name__ == "__main__":
    run_pipeline()


Saved PCA master to: /Users/gauravpathak/Desktop/Bilateral-Trade-Analyais-of-Middle-East-and-Australia/cleaned_trade_master_with_pca.csv
Saved explained variance to: /Users/gauravpathak/Desktop/Bilateral-Trade-Analyais-of-Middle-East-and-Australia/cleaned_trade_master_with_pca_explained_variance.csv
Saved PCA column list to: /Users/gauravpathak/Desktop/Bilateral-Trade-Analyais-of-Middle-East-and-Australia/pca_columns_used.csv
All clustering outputs in: /Users/gauravpathak/Desktop/Bilateral-Trade-Analyais-of-Middle-East-and-Australia/cluster_outputs
 - .DS_Store
 - country_coords_lookup_pca2d.csv
 - country_hier_dendrogram.png
 - country_kmeans_labels.csv
 - country_kmeans_pca2d_top20.png
 - country_percluster_plots
 - country_silhouette.png
 - product_coords_lookup_pca2d.csv
 - product_hier_dendrogram.png
 - product_kmeans_labels.csv
 - product_kmeans_pca2d_top20.png
 - product_percluster_plots
 - product_silhouette.png
 - run_metadata.json
